RLGym TODO
- Add boost locations/other things as needed to obs, like in https://github.com/Rolv-Arild/Necto/blob/2e6ed7d6ed2b352e8ff529d4a12a0c9c70c28cca/training/obs.py#L205
- Implement EARLPerceiver (https://github.com/Rolv-Arild/EARL-pytorch/blob/master/earl_pytorch/model/earl_perceiver.py) and add a query tag to `inputs` to indicate which segments are part of the queries
- Zero pad or add support for multiple sizes in buffer
- Implement previous model loading for the environment to prevent strategy cycling
- Could store each env in buffer, then split per car to parallelize by specifying a splitter segment in config - but removes some observation optimizations, introduces sampling complexity
- Add default MLP handling for envs without args
- Add appropriate sampling with uneven buffer sizes to independent buffer
- First try just storing all permutations of obs to different buffer slots, will use 6x memory but that's the only real downside since we need to use positional embeddings
- Make attention position encoder and decoder

Ordered TODO
- Test or revise continuous actions and TwoHot discretization (don't forget about objective in `train`), rewards sampling is just mean of TwoHot
- Vectorize discrete and discretized actions (or at least allow for concise configuration)
- Try evaluation with no random
- Internally rename library
- Add timers
- Refactor initialization
- Add mixed precision training, or at least fp16
- Implement distributed
- Implement Plan2Explore
- Add eval as parallel env to training loop, or put in another worker
- Add missing tests
- Vectorize buffer storage and sampling

Novel changes
- TwoHot discretization
- Could add variation output to reward, allowing for trust regions and exploration, maybe an auxiliary loss?

Training stage observations
- Random action
- Overfits to one action
- Then explores slowly from there
- Slower to adapt later on - maybe biased sampling?

Commit list
- `make pre-commit`, ensure tests pass, documentation generates, and no linting errors
- Change versions in `pyproject.toml` and `CITATION.cff` - or find a way around this

Running commands
- Run exported notebook: `conda activate fishyrl && cd examples && python Dreamer.py`
- Run tensorboard: `conda activate fishyrl && tensorboard --samples_per_plugin=scalars=99999 --logdir ./examples/runs --host 0.0.0.0`
- Host documentation: `cd ./docs/build/html && python -m http.server 8000`
- Convert gif to loop: `ffmpeg -i <INFILE> -loop 0 -filter_complex "[0:v] fps=30,scale=w=480:h=-1:flags=lanczos,split [a][b];[a] palettegen [p];[b][p] paletteuse" <OUTFILE>`

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import math
import os
import re
import time

import torch
import torch.utils.tensorboard

import fishyrl as frl

os.environ['MUJOCO_GL'] = 'egl'  # MuJoCo rendering backend for headless

I0000 00:00:1777274145.532124  139023 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Initialization

In [7]:
# Load config (NOTE: Times are unreliable here, as they were used in varying workloads)
# cfg = frl.utilities.load_config('../examples/configs/CartPole.yaml', '../examples/configs/Dreamer_Base.yaml')  # Load CartPole on top of Dreamer_Base (CartPole-v1_2026-04-10_19-57-31, 23k steps, 1.6h to full-optimal)
# cfg = frl.utilities.load_config('../examples/configs/LunarLander.yaml', '../examples/configs/Dreamer_Base.yaml')  # (BipedalWalker-v3_2026-04-12_08-12-53)
# cfg = frl.utilities.load_config('../examples/configs/BipedalWalker.yaml', '../examples/configs/Dreamer_Base.yaml')  # (LunarLander-v3_2026-04-12_02-59-29)
# cfg = frl.utilities.load_config('../examples/configs/Ant.yaml', '../examples/configs/Dreamer_Base.yaml')  # (Ant-v5_2026-04-12_14-52-06)
# cfg = frl.utilities.load_config('../examples/configs/Hopper.yaml', '../examples/configs/Dreamer_12m.yaml', '../examples/configs/Dreamer_Base.yaml')  # ()
# cfg = frl.utilities.load_config('../examples/configs/Reacher.yaml', '../examples/configs/Dreamer_1m.yaml', '../examples/configs/Dreamer_Base.yaml')  # (Reacher-v5_2026-04-17_23-38-50)
# cfg = frl.utilities.load_config('../examples/configs/Walker2D.yaml', '../examples/configs/Dreamer_Base.yaml')  # (Walker2d-v5_2026-04-14_02-55-28)
# cfg = frl.utilities.load_config('../examples/configs/Pusher.yaml', '../examples/configs/Dreamer_Base.yaml')  # (Pusher-v5)
# cfg = frl.utilities.load_config('../examples/configs/Humanoid.yaml', '../examples/configs/Dreamer_1m.yaml', '../examples/configs/Dreamer_Base.yaml')  # (Humanoid-v5)
cfg = frl.utilities.load_config('../examples/configs/RLGym.yaml', '../examples/configs/Dreamer_12m.yaml', '../examples/configs/Dreamer_Base.yaml')  # ()

# Checkpoint file if not starting from scratch
checkpoint_file = None
start_environment_step = int(re.search(r'checkpoint_(\d+).pt', checkpoint_file).group(1)) if checkpoint_file is not None else 0

# Folder name if manually specified, otherwise auto-generate
folder_name = None

# Define save folder in same folder as checkpoint if loading, otherwise create new folder with timestamp
if folder_name is None:
    if checkpoint_file is not None:
        folder_name = os.path.dirname(checkpoint_file)
    else:
        folder_name = f'{cfg.name}_{time.strftime("%Y-%m-%d_%H-%M-%S")}'
        folder_name = f'../examples/runs/{folder_name}'
# folder_name = '../examples/runs/TESTING'  # Override folder name for testing

# Load environment and actions
envs = frl.dreamer.construct_envs(cfg=cfg)
actions = frl.dreamer.construct_actions(cfg=cfg)

# Construct models
world_model, actor_critic_model, utility_modules = frl.dreamer.construct_models(
    # env_obs_dim=math.prod(envs.obs_shape[1:]),  # TODO: Maybe readd default env params
    env_num=envs.num_envs,  # Override env_num for cases like RLGym where effective env_num is different from defined env_num
    env_actions=actions,
    device='cuda',
    cfg=cfg,
)

# Load model and buffer
if checkpoint_file is not None:
    frl.dreamer.load_models(
        path=checkpoint_file,
        world_model=world_model,
        actor_critic_model=actor_critic_model,
        utility_modules=utility_modules,
    )

In [8]:
# Count params for testing
wm_params = sum(p.numel() for p in world_model.parameters())
ac_params = sum(p.numel() for p in actor_critic_model.parameters())
print(f'Model has {wm_params + ac_params:,} parameters ({wm_params:,} in world model, {ac_params:,} in actor-critic model)')
components = {
    'encoder': world_model.encoder_model,
    'decoder': world_model.decoder_model,
    'recurrent': world_model.rssm_model._recurrent_model,
    'representation': world_model.rssm_model._representation_model,
    'transition': world_model.rssm_model._transition_model,
    'reward': world_model.reward_model,
    'continue': world_model.continue_model,
    'actor': actor_critic_model.actor_model,
    'critic': actor_critic_model.critic_model,
}
for name, m in components.items():
    p = sum(x.numel() for x in m.parameters())
    print(f"{name:>15}: {p/1e6:.2f}M")

Model has 11,526,584 parameters (9,008,480 in world model, 2,518,104 in actor-critic model)
        encoder: 0.62M
        decoder: 1.27M
      recurrent: 4.36M
 representation: 0.72M
     transition: 0.66M
         reward: 0.72M
       continue: 0.66M
          actor: 0.81M
         critic: 0.85M


In [6]:
# Count params for testing
wm_params = sum(p.numel() for p in world_model.parameters())
ac_params = sum(p.numel() for p in actor_critic_model.parameters())
print(f'Model has {wm_params + ac_params:,} parameters ({wm_params:,} in world model, {ac_params:,} in actor-critic model)')
components = {
    'encoder': world_model.encoder_model,
    'decoder': world_model.decoder_model,
    'recurrent': world_model.rssm_model._recurrent_model,
    'representation': world_model.rssm_model._representation_model,
    'transition': world_model.rssm_model._transition_model,
    'reward': world_model.reward_model,
    'continue': world_model.continue_model,
    'actor': actor_critic_model.actor_model,
    'critic': actor_critic_model.critic_model,
}
for name, m in components.items():
    p = sum(x.numel() for x in m.parameters())
    print(f"{name:>15}: {p/1e6:.2f}M")

Model has 10,688,476 parameters (8,170,372 in world model, 2,518,104 in actor-critic model)
        encoder: 0.23M
        decoder: 0.82M
      recurrent: 4.36M
 representation: 0.72M
     transition: 0.66M
         reward: 0.72M
       continue: 0.66M
          actor: 0.81M
         critic: 0.85M


## Training

In [ ]:
# Tensorboard writer
# TODO: Create folder regardless of writer
writer = torch.utils.tensorboard.SummaryWriter(folder_name)

# Train the model
frl.dreamer.train_loop(
    envs=envs,
    world_model=world_model,
    actor_critic_model=actor_critic_model,
    utility_modules=utility_modules,
    tensorboard_writer=writer,
    start_environment_step=start_environment_step,
    checkpoint_dir=folder_name,
    cfg=cfg,
)

# # Save model
# frl.dreamer.save_models(
#     path=os.path.join(folder_name, 'model.pth'),
#     world_model=world_model,
#     actor_critic_model=actor_critic_model,
#     utility_modules=utility_modules,
# )

# Close tensorboard writer
writer.close()

## Evaluation

In [ ]:
# Evaluate the model
frames, fps = frl.dreamer.evaluate(
    env=envs.copy(num_envs=1, allow_rendering=True),
    world_model=world_model,
    actor_critic_model=actor_critic_model,
    seed=42,
)

# Save video to file with appropriate postfix
step_class = max(0, min(4, math.log10(start_environment_step) // 3)) if start_environment_step > 0 else 0
step_fac_dict = {
    0: (10**0, ''), 1: (10**3, 'k'), 2: (10**6, 'm'), 3: (10**9, 'b'), 4: (10**12, 't')}
step_fac, step_post = step_fac_dict[step_class]
path = f'./images/{cfg.name}_{start_environment_step/1000:.0f}k.gif'
frl.utilities.export_frames(path, frames, fps=fps)

# Display video locally
from IPython.display import Image  # noqa: E402, I001
Image(filename=path)